In [ ]:
import yaml
import sys
sys.path.append("..")
from src.model.finetune import finetune

cfg = yaml.safe_load(open("../config.yaml"))

import warnings
warnings.filterwarnings("ignore", message="Running on CPU with more than")


In [ ]:
finetune(cfg, n_epochs=10, n_surfaces_per_epoch=20, n_context=20)

In [ ]:
import numpy as np
import torch
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
from sklearn.metrics import mean_squared_error
from src.data_generation.data_preperation import data_preparation

N_TEST_SURFACES = 50
N_CONTEXT = 20
N_ESTIMATORS = 1   # match training config

# --- generate held-out surfaces ---
test_train, test_test = data_preparation(cfg, N_TEST_SURFACES, N_CONTEXT)

# --- baseline: stock pre-trained TabPFN V3 (same version as finetuned) ---
baseline = TabPFNRegressor.create_default_for_version(
    version=ModelVersion.V3, n_estimators=N_ESTIMATORS
)

# --- finetuned: V3 with weights replaced by finetuned checkpoint ---
finetuned = TabPFNRegressor.create_default_for_version(
    version=ModelVersion.V3, fit_mode="fit_preprocessors", n_estimators=N_ESTIMATORS
)
finetuned._initialize_model_variables()
finetuned_state = torch.load("../checkpoints/finetune_v1/final.pt", map_location="cpu")
finetuned.model_.load_state_dict(finetuned_state)

# IMPORTANT: TabPFNRegressor.fit() internally calls _initialize_model_variables(),
# which re-downloads the *pretrained* checkpoint and overwrites models_/model_.
# So the finetuned weights must be re-applied after every .fit() call, before .predict().

# --- evaluate both on each held-out surface ---
def eval_surfaces(model, train_list, test_list, reload_state=None):
    rmses, maes, mapes = [], [], []
    for (X_tr, y_tr), (X_te, y_te) in zip(train_list, test_list):
        model.fit(X_tr, y_tr)
        if reload_state is not None:
            model.model_.load_state_dict(reload_state)
        y_pred = model.predict(X_te)
        rmses.append(np.sqrt(mean_squared_error(y_te, y_pred)))
        maes.append(np.mean(np.abs(y_te - y_pred)))
        mapes.append(np.mean(np.abs((y_te - y_pred) / y_te)) * 100)
    return np.mean(rmses), np.mean(maes), np.mean(mapes)

print("Evaluating baseline (V3, pretrained) ...")
b_rmse, b_mae, b_mape = eval_surfaces(baseline, test_train, test_test)
print("Evaluating finetuned (V3) ...")
f_rmse, f_mae, f_mape = eval_surfaces(finetuned, test_train, test_test, reload_state=finetuned_state)

print(f"\n{'':20s} {'Baseline':>12s} {'Finetuned':>12s} {'Delta':>10s}")
print("-" * 56)
for name, b, f in [("RMSE", b_rmse, f_rmse), ("MAE", b_mae, f_mae), ("MAPE (%)", b_mape, f_mape)]:
    print(f"{name:20s} {b:12.4f} {f:12.4f} {f - b:+10.4f}")
